In [9]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

customers_df = spark.read \
    .option("header", "true") \
    .csv("abfss://RetailPulse_DataPlatform@onelake.dfs.fabric.microsoft.com/bronze_lakehouse.Lakehouse/Files/raw/customers/customers_master.csv")

print("Raw customers:", customers_df.count())
customers_df.printSchema()


StatementMeta(, 3dbb90bc-2aaa-4fa4-8482-5e9ff888abc4, 11, Finished, Available, Finished, False)

Raw customers: 500
root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- pincode: string (nullable = true)
 |-- date_joined: string (nullable = true)
 |-- loyalty_tier: string (nullable = true)
 |-- total_orders: string (nullable = true)
 |-- is_active: string (nullable = true)



In [10]:
def parse_active(val):
    if val is None: return False
    return val.strip().lower() in ("y","yes","1","true")

active_udf = F.udf(parse_active, T.BooleanType())

city_map = {
    "mumbai":"Mumbai","mum":"Mumbai","mmbai":"Mumbai",
    "delhi":"Delhi","new delhi":"Delhi","ncr":"Delhi",
    "bangalore":"Bangalore","bengaluru":"Bangalore","blr":"Bangalore",
    "hyderabad":"Hyderabad","hyd":"Hyderabad",
    "chennai":"Chennai","pune":"Pune",
    "kolkata":"Kolkata","calcutta":"Kolkata",
    "ahmedabad":"Ahmedabad","jaipur":"Jaipur","surat":"Surat"
}
def clean_city(c):
    if c is None: return "Unknown"
    return city_map.get(c.lower().strip(), c.strip().title())

city_udf = F.udf(clean_city, T.StringType())

cleaned_df = customers_df \
    .withColumn("date_joined",   F.to_date(F.col("date_joined"))) \
    .withColumn("total_orders",  F.col("total_orders").cast(T.IntegerType())) \
    .withColumn("is_active",     active_udf(F.col("is_active"))) \
    .withColumn("city",          city_udf(F.col("city"))) \
    .withColumn("state",         F.upper(F.trim(F.col("state")))) \
    .withColumn(
        # Remove impossible negative values
        "total_orders",
        F.when(F.col("total_orders") < 0, F.lit(None))
         .otherwise(F.col("total_orders"))
    ) \
    .withColumn(
        "is_valid_email",
        F.col("email").rlike(r"^[^@]+@[^@]+\.[^@]+$")
    ) \
    .withColumn(
        "days_as_customer",
        F.datediff(F.current_date(), F.col("date_joined"))
    ) \
    .withColumn("silver_created_at", F.current_timestamp())

print("Cleaned customers:", cleaned_df.count())

StatementMeta(, 3dbb90bc-2aaa-4fa4-8482-5e9ff888abc4, 12, Finished, Available, Finished, False)

Cleaned customers: 500


In [11]:
# Drop only truly bad rows — missing customer_id
good_df = cleaned_df.filter(F.col("customer_id").isNotNull())

# FIX: Define the fully qualified 3-part name (catalog.schema.table)
TARGET = "silver_customers"

good_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_customers")

print("Done:", spark.sql("SELECT COUNT(*) FROM silver_customers").collect()[0][0])

StatementMeta(, 3dbb90bc-2aaa-4fa4-8482-5e9ff888abc4, 13, Finished, Available, Finished, False)

Done: 500
